# Extending `src.xai_adapter` With a New XAI Method

This notebook is a companion to [`xai_adapter_quickstart.ipynb`](xai_adapter_quickstart.ipynb).
That notebook shows how to **use** the methods that already ship with `src.xai_adapter`
(SHAP, LOFO, surrogate methods, ...). This one shows how to **author a brand-new one**.

Sections:
1. The adapter contract — `XAIAdapter`, `XAIAdapterResult`, `LocalAttribution`
2. Define a new adapter — `GroupAblationAttribution`
3. Smoke test it directly (no registry involved)
4. Register it for ad-hoc use — `register_xai_method`
5. Optional: promote it to a first-class built-in in `registry.py`
6. Optional: make it selectable from an experiment design (`support_matrix.json`)
7. Wire it into the simulation workflow (`generate_xai_explanation_tables`, `study.run_experiment`)
8. A test you can drop into `tests/`

In [1]:
from pathlib import Path
import sys

# Support running from repo root or from the tutorials/ subfolder
repo_root = Path.cwd().parent if Path.cwd().name == "tutorials" else Path.cwd()
sys.path.insert(0, str(repo_root))

import inspect
from typing import Callable, List, Optional

import numpy as np
import pandas as pd

from src.xai_adapter import (
    XAIAdapter,
    XAIAdapterResult,
    baseline_from_data,
    create_xai_method,
    ensure_2d,
    get_adapter_registry,
    register_xai_method,
    select_target,
)
from src.xai_adapter.attribution import LeaveOneFeatureOut, LocalAttribution

print("repo_root:", repo_root)

repo_root: /Users/wangzhuoyulucas/Documents/GitHub/xaikit-test-api


## 1 · The adapter contract

Every XAI method in this project is a subclass of `XAIAdapter`
(`src/xai_adapter/base.py`). The ABC is deliberately small — one abstract method:

In [2]:
print(inspect.getsource(XAIAdapter.explain))
print(inspect.getsource(XAIAdapter.fit))

    @abstractmethod
    def explain(self, instances: ArrayLike) -> XAIAdapterResult:
        """Explain one or more instances."""

    def fit(self, X: ArrayLike = None, y: ArrayLike = None, **kwargs):
        """Fit or initialize method state. Subclasses override when needed."""
        self.is_fitted = True
        return self



What a new adapter needs to provide:

| Method | Required? | Purpose |
|---|---|---|
| `explain(instances) -> XAIAdapterResult` | **Yes** (abstract) | Compute attributions for a batch of instances |
| `fit(X=None, y=None, **kwargs)` | No (default just sets `is_fitted = True`) | Fit/initialize method state from background or training data |
| `__init__` | No fixed shape | Whatever kwargs the method needs — see `KernelShap` (`predict_fn`) vs `ShapTreeExplainer` (`model`) for how loosely this is held |

`explain()` must return an `XAIAdapterResult` — the dataclass every downstream consumer
(plotting, CSV export, the experiment pipeline) expects:

```python
@dataclass
class XAIAdapterResult:
    values: np.ndarray        # shape (n_instances, n_features), signed attributions
    base_values: np.ndarray   # shape (n_instances,), per-instance intercept
    method: str
    metadata: Dict[str, Any] = field(default_factory=dict)
```

If your method is a **local, instance-level feature-attribution** method (the common case —
SHAP, LIME, LOFO, ...), subclass `LocalAttribution`
(`src/xai_adapter/attribution/base.py`) instead of `XAIAdapter` directly. It adds nothing
except an `attribute()` alias for `explain()`, but it's how the registry and the rest of the
codebase distinguish "local attribution" methods from surrogate / example-based / concept
methods. Non-attribution kinds (counterfactuals, prototypes, surrogates, TCAV) subclass
`XAIAdapter` directly instead — see `src/xai_adapter/example_based/`, `src/xai_adapter/surrogate/`,
`src/xai_adapter/concept/` for those patterns.

Helper functions available to any adapter (all in `base.py`):
`identity_preprocess`, `identity_postprocess`, `ensure_2d`, `select_target`, `baseline_from_data`.

## 2 · Define a new adapter: `GroupAblationAttribution`

The simplest existing method to build on is `LeaveOneFeatureOut` (LOFO,
`src/xai_adapter/attribution/perturbation.py`) — it needs no external library, just a
`predict_fn`. LOFO ablates **one feature at a time**. That's a problem for one-hot encoded
or otherwise correlated feature groups: ablating a single dummy column barely moves the
prediction, so the whole group looks unimportant even when the underlying categorical
variable matters a lot.

`GroupAblationAttribution` fixes that: it ablates a **group** of columns together, then
splits the resulting delta evenly across the group's members. Passing `feature_groups=None`
falls back to one-feature-per-group, i.e. plain LOFO — so it's a strict generalization.

This is deliberately the smallest possible diff from `LeaveOneFeatureOut` so the pattern
you'd copy for your *own* new method is easy to see: same constructor shape, same
`fit()`/`explain()` split, same use of `ensure_2d` / `select_target` / `baseline_from_data` /
`_postprocess_values`.

In [3]:
class GroupAblationAttribution(LocalAttribution):
    """Local attribution via grouped feature ablation.

    Ablates one feature group at a time (replacing it with a baseline value),
    measures the resulting change in the target output, and splits that delta
    evenly across the group's member features. Passing ``feature_groups=None``
    degenerates to plain leave-one-feature-out.
    """

    method_name = "group_ablation"

    def __init__(
        self,
        *,
        predict_fn: Callable[[np.ndarray], np.ndarray],
        feature_groups: Optional[List[List[int]]] = None,
        background_data=None,
        baseline: str = "mean",
        target: int = 1,
        preprocessing_fn=None,
        postprocessing_fn=None,
    ):
        super().__init__(
            target=target,
            preprocessing_fn=preprocessing_fn,
            postprocessing_fn=postprocessing_fn,
        )
        self.predict_fn = predict_fn
        self.feature_groups = feature_groups
        self.baseline = baseline
        self.baseline_vec = None
        if background_data is not None:
            self.fit(background_data)

    def fit(self, X, y=None, **kwargs):
        """Fit the replacement baseline from background data."""
        x = ensure_2d(self.preprocessing_fn(X))
        self.baseline_vec = baseline_from_data(x, self.baseline)
        if self.feature_groups is None:
            self.feature_groups = [[i] for i in range(x.shape[1])]
        self.is_fitted = True
        return self

    def explain(self, instances) -> XAIAdapterResult:
        self._require_fitted()
        raw_instances = ensure_2d(instances)
        x = ensure_2d(self.preprocessing_fn(raw_instances))
        base_probs = select_target(self.predict_fn(x), self.target)

        attributions = np.zeros_like(x, dtype=float)
        for group in self.feature_groups:
            masked = x.copy()
            masked[:, group] = self.baseline_vec[group]
            masked_probs = select_target(self.predict_fn(masked), self.target)
            delta = base_probs - masked_probs
            for idx in group:
                attributions[:, idx] = delta / len(group)

        values = self._postprocess_values(raw_instances, attributions)
        return XAIAdapterResult(
            values=values,
            base_values=np.full(values.shape[0], float(np.mean(base_probs)), dtype=float),
            method=self.method_name,
            metadata={"feature_groups": self.feature_groups, "baseline": self.baseline_vec},
        )

## 3 · Smoke test — compare against plain LOFO

Toy setup: two one-hot dummy columns encoding a 2-level categorical (`color_red`,
`color_blue`) plus one continuous feature. The categorical drives the prediction; LOFO
should under-attribute each dummy column on its own, while grouping them recovers the
combined effect.

In [4]:
def predict_proba(X):
    X = np.asarray(X, dtype=float)
    # color (columns 0,1 one-hot) dominates; column 2 is a weak continuous signal
    p1 = np.clip(0.15 + 0.6 * X[:, 0] + 0.05 * X[:, 2], 0.0, 1.0)
    return np.column_stack([1.0 - p1, p1])


X_train = np.array([
    [1.0, 0.0, 0.2],
    [0.0, 1.0, 0.4],
    [1.0, 0.0, 0.6],
    [0.0, 1.0, 0.8],
])
X_test = np.array([[1.0, 0.0, 0.5]])
FEATURE_NAMES = ["color_red", "color_blue", "continuous"]

lofo = LeaveOneFeatureOut(predict_fn=predict_proba, background_data=X_train, target=1)
lofo_result = lofo.explain(X_test)

grouped = GroupAblationAttribution(
    predict_fn=predict_proba,
    feature_groups=[[0, 1], [2]],  # color dummies grouped, continuous feature alone
    background_data=X_train,
    target=1,
)
grouped_result = grouped.explain(X_test)

comparison = pd.DataFrame(
    np.vstack([lofo_result.values[0], grouped_result.values[0]]),
    columns=FEATURE_NAMES,
    index=["lofo (per-column)", "group_ablation (color grouped)"],
)
comparison

,color_red,color_blue,continuous
lofo (per-column),0.30,0.00,0.0
group_ablation (color grouped),0.15,0.15,0.0


Per-column LOFO splits the color signal thinly across `color_red`/`color_blue`, while
`group_ablation` attributes the full swing to the `color` group and splits it evenly
between its two dummy columns — a more honest picture of "how much did color matter."

## 4 · Register it for ad-hoc use

You don't need to touch the package to use a new adapter in a notebook or an experiment
script — `register_xai_method` (`src/xai_adapter/registry.py:482`) adds it to the global
registry at runtime:

In [5]:
register_xai_method("group_ablation", GroupAblationAttribution, "grouped_lofo")

print("group_ablation" in get_adapter_registry().list_available())
print("grouped_lofo" in get_adapter_registry().list_available())

# Now it's usable the same way every built-in method is: through create_xai_method(...)
via_registry = create_xai_method(
    "group_ablation",
    predict_fn=predict_proba,
    feature_groups=[[0, 1], [2]],
    background_data=X_train,
    target=1,
)
via_registry.explain(X_test).values

True
True


array([[0.15, 0.15, 0.  ]])

## 5 · Optional: promote it to a first-class built-in

Ad-hoc registration (section 4) is enough for a notebook or a one-off experiment script.
If the method should ship with the package so every caller gets it for free, add it to
`get_adapter_registry()` in `src/xai_adapter/registry.py` — the pattern is the same one
line per method that every existing adapter uses:

```python
# src/xai_adapter/registry.py, inside get_adapter_registry()
from .attribution import (
    ...,
    GroupAblationAttribution,  # 1. import your class alongside the others
)
...
registry.register("group_ablation", GroupAblationAttribution, "grouped_lofo")  # 2. one line
```

And re-export it from the package's public surface, matching every other adapter in
`src/xai_adapter/__init__.py`:

```python
from .attribution import (
    ...,
    GroupAblationAttribution,
)
__all__ = [
    ...,
    "GroupAblationAttribution",
]
```

(and the mirrored entries in `src/xai_adapter/attribution/__init__.py` for the
`attribution` subpackage). This cell block is illustrative, not executed — editing package
source from inside a notebook isn't something you want happening implicitly.

## 6 · Optional: make it selectable from an experiment design

If the method should be choosable as an `xai_method` level when building experiment
designs, add it to `groups.xai_methods.<framework>` in
`src/experiment_planner/support_matrix.json` — this is the file `support.py` validates
designs against (`_check_allowed(..., "xai_method", ...)`):

```json
"xai_methods": {
  "coax": [
    "none", "lime", "shap", "integrated_gradients", "input_gradients", "lrp", "deeplift",
    "group_ablation"
  ],
  ...
}
```

`"supported"` already references `$groups.xai_methods.coax`, so nothing else needs to
change for the value to become valid. Only bother with
`XAI_METHOD_ALIASES` / `IV_FACTOR_ALIASES` in `src/experiment_planner/design_export.py`
if the design UI writes a prose name for the method that doesn't already slugify to
`group_ablation`.

## 7 · Wire it into the simulation workflow

A simulated study (`xaikitTest` in `src/api.py`, driven end-to-end in
`tutorials/feature_explanation_user_study.ipynb`) never talks to `create_xai_method`
directly. The chain is:

```
study.add_iv('xai_method', 'between', ['none', 'lime', 'group_ablation'])
          |
          v
init_explanation_run(data, iv_config, trained_ai_model, ...)
          |
          v
generate_xai_explanation_tables(config)          # src/xai_adapter/api.py:264
  -> get_xai_methods_from_design(config.iv_config)  # reads the 'xai_method' IV levels
  -> for each level: _make_explainer(method_key, config, train_data_for_xai)  # api.py:213
       -> create_xai_method(method_key, ai_model=config.trained_ai_model,
                             train_data=train_data_for_xai, target=config.target, ...)
  -> explainer.explain(...) -> .to_explanation_df(...) -> CSV
          |
          v
study.run_experiment(mode='whole_experiment', ...)   # cognitive model consumes the CSV
```

So as soon as `group_ablation` is (a) registered — section 4 — and (b) a valid
`xai_method` level — section 6 — it's a legal IV level. But `_make_explainer` always
calls `create_xai_method` with `ai_model=`/`train_data=`, never `predict_fn=`
directly, so the adapter also has to accept the model through that path.

`ai_model=`/`train_data=` are translated into constructor kwargs by
`_apply_ai_model_kwargs` (`src/xai_adapter/registry.py:168`), which only knows about a
fixed set of registry keys. Watch what happens for `group_ablation`, which isn't one of
them:

In [6]:
class FakeAIModel:
    """Stand-in for a trained_ai_model: exposes .predict like the real thing."""
    def __init__(self, predict_fn):
        self.predict = predict_fn
        self.model = None


class FakeTrainData:
    """Stand-in for the train_data_for_xai object: exposes .X like the real thing."""
    def __init__(self, X):
        self.X = X


fake_model = FakeAIModel(predict_proba)
fake_train = FakeTrainData(X_train)

try:
    create_xai_method("group_ablation", ai_model=fake_model, train_data=fake_train, target=1)
except TypeError as exc:
    print("Fails without wiring:", exc)

Fails without wiring: GroupAblationAttribution.__init__() missing 1 required keyword-only argument: 'predict_fn'


`GroupAblationAttribution` wants exactly what LOFO wants — `predict_fn` and
`background_data` — so the fix is adding it to the same branch LOFO/SHAP already use in
`_apply_ai_model_kwargs`:

```python
# src/xai_adapter/registry.py, inside _apply_ai_model_kwargs
if key in {"lofo", "leave_one_feature_out", "shap", "shap_kernel", "group_ablation", "grouped_lofo"}:
    kwargs.update(predict_fn=predict_fn, background_data=train_x)
```

That's a real source edit (see section 5). To prove the fix without editing the checked-out
file, patch the same function at runtime and re-run the failing call:

In [7]:
import src.xai_adapter.registry as _registry

_original_apply_ai_model_kwargs = _registry._apply_ai_model_kwargs


def _patched_apply_ai_model_kwargs(key, ai_model, train_data, kwargs):
    if key in {"group_ablation", "grouped_lofo"}:
        kwargs.setdefault("predict_fn", ai_model.predict)
        kwargs.setdefault("background_data", _registry._train_x(train_data))
        return kwargs
    return _original_apply_ai_model_kwargs(key, ai_model, train_data, kwargs)


_registry._apply_ai_model_kwargs = _patched_apply_ai_model_kwargs

wired = create_xai_method(
    "group_ablation",
    ai_model=fake_model,
    train_data=fake_train,
    feature_groups=[[0, 1], [2]],
    target=1,
)
wired.explain(X_test).values

array([[0.15, 0.15, 0.  ]])

That's exactly the call `_make_explainer` makes inside `generate_xai_explanation_tables`
during a real simulated study — once the one-line branch above is merged into
`registry.py`, no monkeypatching is needed and `group_ablation` behaves like any other
built-in method for `study.run_experiment(...)`.

## 8 · A test to drop into `tests/`

There's no `tests/test_<adapter>.py` convention yet in this repo (the one that exists,
`tests/test_sim2real_data_loader.py`, tests the precomputed-CSV adapter specifically). The
closest thing to a template is this notebook's own smoke test in section 3 — turned into
assertions:

In [8]:
def test_group_ablation_attribution_shapes_and_values():
    predict_fn = predict_proba
    X_bg = X_train
    X_query = X_test

    adapter = GroupAblationAttribution(
        predict_fn=predict_fn,
        feature_groups=[[0, 1], [2]],
        background_data=X_bg,
        target=1,
    )
    result = adapter.explain(X_query)

    assert isinstance(result, XAIAdapterResult)
    assert result.values.shape == (X_query.shape[0], X_query.shape[1])
    assert result.base_values.shape == (X_query.shape[0],)
    assert result.method == "group_ablation"

    # grouped color attribution should split evenly across the two dummies
    assert np.isclose(result.values[0, 0], result.values[0, 1])

    # feature_groups=None degenerates to plain per-feature LOFO
    ungrouped = GroupAblationAttribution(predict_fn=predict_fn, background_data=X_bg, target=1)
    ungrouped_result = ungrouped.explain(X_query)
    lofo_reference = LeaveOneFeatureOut(predict_fn=predict_fn, background_data=X_bg, target=1)
    np.testing.assert_allclose(ungrouped_result.values, lofo_reference.explain(X_query).values)

    print("OK")


test_group_ablation_attribution_shapes_and_values()

OK


## Checklist for adding a new XAI method

1. Subclass `LocalAttribution` (attribution methods) or `XAIAdapter` directly (everything
   else) in a new or existing file under `src/xai_adapter/`.
2. Implement `explain(instances) -> XAIAdapterResult`; override `fit()` if the method needs
   background/training data.
3. Smoke-test it directly, unregistered — instantiate and call `.explain()`.
4. `register_xai_method("your_name", YourClass, *aliases)` for ad-hoc use, **or** add an
   import + `registry.register(...)` line in `get_adapter_registry()`
   (`src/xai_adapter/registry.py`) to ship it as a built-in, plus the matching re-export in
   `src/xai_adapter/__init__.py` (and `attribution/__init__.py` if applicable).
5. If it should be selectable from an experiment design, add its name to
   `groups.xai_methods.<framework>` in `src/experiment_planner/support_matrix.json`.
6. If it should run inside a simulated study (`study.run_experiment(...)`), add its
   registry key(s) to the matching branch in `_apply_ai_model_kwargs`
   (`src/xai_adapter/registry.py:168`) so `ai_model=`/`train_data=` resolve to the kwargs
   your `__init__` expects.
7. Add a test function in the shape of section 8.